In [3]:
import json
import random
import time
from datetime import datetime, timedelta

from kafka import KafkaProducer
from kafka.errors import KafkaError, NoBrokersAvailable


# =========================
# KONFIGURACJA KAFKA
# =========================

KAFKA_BOOTSTRAP_SERVERS = "broker:9092"
TOPIC_NAME = "restaurant_orders"


# =========================
# MENU RESTAURACJI
# =========================
# Czasy przygotowania są celowo skrócone do sekund,
# żeby podczas demo było widać zmianę statusów zamówień na dashboardzie.

MENU = [
    {
        "dish_name": "Pizza Margherita",
        "dish_category": "Pizza",
        "price": 29.90,
        "expected_preparation_seconds": 60
    },
    {
        "dish_name": "Pizza Pepperoni",
        "dish_category": "Pizza",
        "price": 34.90,
        "expected_preparation_seconds": 70
    },
    {
        "dish_name": "Pizza Capricciosa",
        "dish_category": "Pizza",
        "price": 36.90,
        "expected_preparation_seconds": 75
    },
    {
        "dish_name": "Burger klasyczny",
        "dish_category": "Burger",
        "price": 31.90,
        "expected_preparation_seconds": 40
    },
    {
        "dish_name": "Burger BBQ",
        "dish_category": "Burger",
        "price": 36.90,
        "expected_preparation_seconds": 45
    },
    {
        "dish_name": "Makaron Carbonara",
        "dish_category": "Makaron",
        "price": 32.90,
        "expected_preparation_seconds": 50
    },
    {
        "dish_name": "Makaron Bolognese",
        "dish_category": "Makaron",
        "price": 34.90,
        "expected_preparation_seconds": 55
    },
    {
        "dish_name": "Sałatka Cezar",
        "dish_category": "Sałatka",
        "price": 27.90,
        "expected_preparation_seconds": 25
    },
    {
        "dish_name": "Zupa dnia",
        "dish_category": "Zupa",
        "price": 18.90,
        "expected_preparation_seconds": 20
    },
    {
        "dish_name": "Frytki",
        "dish_category": "Dodatek",
        "price": 12.90,
        "expected_preparation_seconds": 15
    }
]


ORDER_CHANNELS = ["app", "website", "phone", "restaurant"]


# =========================
# POŁĄCZENIE Z KAFKA
# =========================

def create_producer():
    """
    Tworzy producenta Kafka, który wysyła wiadomości JSON
    do topicu restaurant_orders.
    """
    try:
        print(f"Próba połączenia z Kafka: {KAFKA_BOOTSTRAP_SERVERS}")

        producer = KafkaProducer(
            bootstrap_servers=[KAFKA_BOOTSTRAP_SERVERS],
            value_serializer=lambda value: json.dumps(
                value,
                ensure_ascii=False
            ).encode("utf-8"),
            api_version=(2, 8, 0),
            request_timeout_ms=30000,
            max_block_ms=30000,
            retries=3
        )

        print("Połączono z Kafka.")
        return producer

    except NoBrokersAvailable:
        print("Nie znaleziono brokera Kafka.")
        print("Sprawdź, czy Kafka działa oraz czy adres bootstrap servera jest poprawny.")
        raise

    except Exception as e:
        print(f"Wystąpił błąd podczas tworzenia producenta Kafka: {e}")
        raise


# =========================
# LOGIKA POPYTU I PROMOCJI
# =========================

def is_promotion_active():
    """
    Określa, czy aktualnie trwa promocja.

    W tej symulacji promocja jest aktywna codziennie w godzinach 18:00-20:59.
    Dzięki temu pole is_promotion zależy od czasu, a nie jest losowe.
    """
    current_hour = datetime.now().hour
    return 18 <= current_hour <= 20


def get_demand_multiplier():
    """
    Określa natężenie ruchu w restauracji.

    Im większy mnożnik, tym częściej producer generuje zamówienia.

    Uwzględniono:
    - większy ruch w porze lunchu,
    - większy ruch w porze kolacji,
    - mniejszy ruch w nocy,
    - dodatkowy wzrost ruchu podczas promocji.
    """
    current_hour = datetime.now().hour
    multiplier = 1.0

    # Lunch
    if 12 <= current_hour <= 14:
        multiplier *= 1.8

    # Kolacja
    elif 18 <= current_hour <= 21:
        multiplier *= 2.2

    # Noc / bardzo niski ruch
    elif current_hour >= 22 or current_hour <= 6:
        multiplier *= 0.4

    # Promocja dodatkowo zwiększa popyt
    if is_promotion_active():
        multiplier *= 1.3

    return multiplier


def get_sleep_time():
    """
    Oblicza, ile sekund producer czeka przed wygenerowaniem kolejnego zamówienia.

    Przy większym popycie czas oczekiwania jest krótszy.
    """
    demand_multiplier = get_demand_multiplier()

    base_sleep = random.uniform(2, 5)
    sleep_time = base_sleep / demand_multiplier

    # Minimalna przerwa, żeby producer nie wysyłał wiadomości zbyt szybko
    return max(0.5, sleep_time)


# =========================
# GENEROWANIE ZAMÓWIENIA
# =========================

def choose_dish():
    """
    Losuje danie z menu.
    Najpopularniejsze dania mają większą wagę.
    """
    weights = [
        0.14,  # Pizza Margherita
        0.16,  # Pizza Pepperoni
        0.10,  # Pizza Capricciosa
        0.14,  # Burger klasyczny
        0.12,  # Burger BBQ
        0.10,  # Makaron Carbonara
        0.08,  # Makaron Bolognese
        0.07,  # Sałatka Cezar
        0.04,  # Zupa dnia
        0.05   # Frytki
    ]

    return random.choices(MENU, weights=weights, k=1)[0]


def generate_quantity():
    """
    Losuje liczbę sztuk w zamówieniu.
    Najczęściej klient zamawia 1 sztukę, rzadziej 2 lub 3.
    """
    return random.choices(
        population=[1, 2, 3],
        weights=[0.65, 0.25, 0.10],
        k=1
    )[0]


def generate_actual_preparation_seconds(expected_seconds):
    """
    Generuje rzeczywisty czas przygotowania zamówienia w sekundach.

    Na potrzeby demo czasy są skrócone.
    Przy większym popycie częściej pojawiają się opóźnienia.
    """
    demand_multiplier = get_demand_multiplier()

    if demand_multiplier >= 2.0:
        delay = random.randint(-5, 25)
    else:
        delay = random.randint(-8, 15)

    actual_seconds = expected_seconds + delay

    # Minimum 5 sekund, żeby nie powstały absurdalnie krótkie lub ujemne czasy
    return max(5, actual_seconds)


def generate_order(order_number):
    """
    Generuje jedno zamówienie restauracyjne.

    Producer wysyła zamówienie z informacją:
    - kiedy powstało,
    - ile powinno się przygotowywać,
    - kiedy według planu będzie gotowe,
    - kiedy faktycznie będzie gotowe,
    - czy jest opóźnione.
    """
    dish = choose_dish()
    quantity = generate_quantity()

    created_at = datetime.now()

    expected_seconds = dish["expected_preparation_seconds"]
    actual_seconds = generate_actual_preparation_seconds(expected_seconds)

    estimated_ready_at = created_at + timedelta(seconds=expected_seconds)
    actual_ready_at = created_at + timedelta(seconds=actual_seconds)

    is_delayed = actual_seconds > expected_seconds
    delay_seconds = max(0, actual_seconds - expected_seconds)

    promotion_active = is_promotion_active()

    order = {
        "order_id": f"ORD_{order_number:06d}",

        # Czas utworzenia zamówienia
        "timestamp": created_at.isoformat(timespec="seconds"),
        "created_at": created_at.isoformat(timespec="seconds"),

        # Dane o produkcie
        "dish_name": dish["dish_name"],
        "dish_category": dish["dish_category"],

        # Dane sprzedażowe
        "quantity": quantity,
        "unit_price": dish["price"],
        "order_value": round(dish["price"] * quantity, 2),
        "order_channel": random.choice(ORDER_CHANNELS),

        # Czasy przygotowania w sekundach
        "expected_preparation_seconds": expected_seconds,
        "actual_preparation_seconds": actual_seconds,

        # Planowany i faktyczny czas zakończenia zamówienia
        "estimated_ready_at": estimated_ready_at.isoformat(timespec="seconds"),
        "actual_ready_at": actual_ready_at.isoformat(timespec="seconds"),

        # Informacja o opóźnieniu
        "is_delayed": is_delayed,
        "delay_seconds": delay_seconds,

        # Status początkowy zamówienia
        # Dashboard może później dynamicznie pokazywać:
        # in_progress, jeśli actual_ready_at > aktualny czas,
        # completed, jeśli actual_ready_at <= aktualny czas.
        "status": "in_progress",

        # Informacja, czy zamówienie powstało podczas promocji
        "is_promotion": promotion_active
    }

    return order


# =========================
# WYSYŁANIE DO KAFKA
# =========================

def send_order(producer, order):
    """
    Wysyła pojedyncze zamówienie do topicu Kafka.
    """
    try:
        future = producer.send(TOPIC_NAME, value=order)
        record_metadata = future.get(timeout=30)

        # Skrócony log w notebooku/terminalu.
        # Pełna wiadomość JSON jest wysyłana do Kafka jako obiekt order.
        print(
            f"Wysłano {order['order_id']} | "
            f"{order['dish_name']} | "
            f"czas={order['actual_preparation_seconds']}s | "
            f"gotowe_o={order['actual_ready_at']} | "
            f"opóźnienie={order['is_delayed']} | "
            f"promotion={order['is_promotion']} | "
            f"topic={record_metadata.topic}, "
            f"partition={record_metadata.partition}, "
            f"offset={record_metadata.offset}"
        )

    except KafkaError as e:
        print("Błąd Kafka podczas wysyłania wiadomości:")
        print(e)
        raise

    except Exception as e:
        print("Inny błąd podczas wysyłania wiadomości:")
        print(e)
        raise


# =========================
# GŁÓWNA PĘTLA PROGRAMU
# =========================

def main():
    producer = create_producer()
    order_number = 1

    print("Producer uruchomiony.")
    print(f"Wysyłanie zamówień do topicu: {TOPIC_NAME}")
    print("Aby zatrzymać producer, użyj Kernel -> Interrupt albo Ctrl+C.")

    try:
        while True:
            order = generate_order(order_number)
            send_order(producer, order)

            order_number += 1

            sleep_time = get_sleep_time()
            time.sleep(sleep_time)

    except KeyboardInterrupt:
        print("Producer został zatrzymany przez użytkownika.")

    finally:
        producer.flush()
        producer.close()
        print("Połączenie z Kafka zostało zamknięte.")


if __name__ == "__main__":
    main()

Próba połączenia z Kafka: broker:9092
Połączono z Kafka.
Producer uruchomiony.
Wysyłanie zamówień do topicu: restaurant_orders
Aby zatrzymać producer, użyj Kernel -> Interrupt albo Ctrl+C.
Wysłano ORD_000001 | Burger klasyczny | czas=57s | gotowe_o=2026-06-01T19:55:26 | opóźnienie=True | promotion=True | topic=restaurant_orders, partition=0, offset=33
Wysłano ORD_000002 | Makaron Bolognese | czas=61s | gotowe_o=2026-06-01T19:55:31 | opóźnienie=True | promotion=True | topic=restaurant_orders, partition=0, offset=34
Wysłano ORD_000003 | Pizza Pepperoni | czas=70s | gotowe_o=2026-06-01T19:55:41 | opóźnienie=False | promotion=True | topic=restaurant_orders, partition=0, offset=35
Wysłano ORD_000004 | Makaron Carbonara | czas=46s | gotowe_o=2026-06-01T19:55:18 | opóźnienie=False | promotion=True | topic=restaurant_orders, partition=0, offset=36
Wysłano ORD_000005 | Pizza Pepperoni | czas=84s | gotowe_o=2026-06-01T19:55:57 | opóźnienie=True | promotion=True | topic=restaurant_orders, partiti